# A network from an exported state dict

A soil classifier was trained in PyTorch and exported to JSON so that a field device with
only NumPy can run it. The device needs the network's three class probabilities for each
reading. You will turn the exported file into the layers of this lesson's forward pass and
run it on five readings, whose probabilities the original PyTorch model also recorded.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It rebuilds the contents of the exported file as the dictionary
`state`, and prints its keys.

In [ ]:
import json
import numpy as np

# A soil-moisture classifier trained in PyTorch as
#   nn.Sequential(Linear(5, 12), ReLU(), Linear(12, 12), ReLU(), Linear(12, 12), ReLU(),
#                 Linear(12, 12), ReLU(), Linear(12, 12), ReLU(), Linear(12, 3))
# was exported with json.dump(model.state_dict() as lists, f, sort_keys=True).
# This cell rebuilds that file's contents.
_rng = np.random.default_rng(61)
_sizes = [5, 12, 12, 12, 12, 12, 3]
_export = {}
for _i, (_n_in, _n_out) in enumerate(zip(_sizes[:-1], _sizes[1:])):
    _export[f"net.{2 * _i}.weight"] = np.round(_rng.normal(0, 2.2 / np.sqrt(_n_in), (_n_out, _n_in)), 4).tolist()
    _export[f"net.{2 * _i}.bias"] = np.round(_rng.normal(0, 0.5, _n_out), 4).tolist()
state = json.loads(json.dumps(_export, sort_keys=True))

# Five readings from the field, one per row: moisture, temperature, salinity, pH, light.
X_field = np.array([
    [0.31, 0.72, 0.15, 0.55, 0.90],
    [0.82, 0.10, 0.64, 0.47, 0.22],
    [0.05, 0.43, 0.91, 0.38, 0.67],
    [0.58, 0.58, 0.27, 0.95, 0.04],
    [0.99, 0.86, 0.73, 0.12, 0.51],
])
print(list(state))

## Part 1 (coding task): The layers in the lesson's layout

Write `layers_from_state(state)`. It returns a list with one `(W, b)` pair per `nn.Linear`
layer, in the order the network applies them, where `W` is a float array of shape
`(n_in, n_out)` as in the lesson's `Z = A @ W + b`, and `b` is a float array of shape
`(n_out,)`.

Every key has the form `<name>.<position>.weight` or `<name>.<position>.bias`, where the
position is the layer's index inside `nn.Sequential`. Your function should work for any
such export, not only this one.

In [ ]:
def layers_from_state(state):
    """List of (W, b) in the order the network applies them, W of shape (n_in, n_out)."""
    # TODO
    raise NotImplementedError

In [ ]:
layers = layers_from_state(state)
assert len(layers) == 6, f"expected 6 layers, got {len(layers)}"
shapes = [(W.shape, b.shape) for W, b in layers]
want = [((5, 12), (12,))] + [((12, 12), (12,))] * 4 + [((12, 3), (3,))]
for l, (got, exp) in enumerate(zip(shapes, want), start=1):
    assert got == exp, f"layer {l}: W and b have shapes {got}, expected {exp}. Check the order of the layers and the layout of W"
assert np.allclose(layers[1][0], np.asarray(state["net.2.weight"]).T), "layer 2 should be net.2, with its weight transposed"
assert np.allclose(layers[5][1], state["net.10.bias"]), "layer 6 should use net.10's bias"
assert np.allclose(layers[5][0], np.asarray(state["net.10.weight"]).T), "layer 6 should be net.10, with its weight transposed"
assert all(W.dtype == float and b.dtype == float for W, b in layers), "W and b should be float arrays"

small = {"enc.3.bias": [0.5], "enc.3.weight": [[1.0, -1.0]],
         "enc.0.weight": [[1.0, 2.0, 0.0], [0.0, 1.0, 1.0]], "enc.0.bias": [0.0, -1.0]}
L = layers_from_state(small)
assert [W.shape for W, _ in L] == [(3, 2), (2, 1)], "positions need not be consecutive: enc.0 comes before enc.3"
assert np.allclose(L[0][0], [[1, 0], [2, 1], [0, 1]]), "the weight of enc.0 should be transposed"
print("Part 1 passed")

## Part 2 (coding task): The probabilities for a batch

Write `predict_proba(X, layers)` for a batch `X` with one reading per row. It returns the
softmax of the network's output: one row of class probabilities per reading.

The check compares your output with the probabilities the PyTorch model recorded for the
five readings in `X_field`.

In [ ]:
def predict_proba(X, layers):
    """Class probabilities, one row per row of X."""
    # TODO
    raise NotImplementedError

In [ ]:
toy = [(np.eye(2), np.zeros(2)), (np.array([[1.0, -1.0], [0.0, 2.0]]), np.array([0.0, -1.0]))]
P_toy = predict_proba(np.array([[1.0, -2.0]]), toy)
assert np.shape(P_toy) == (1, 2), f"expected shape (1, 2) for one reading and two classes, got {np.shape(P_toy)}"
assert np.allclose(P_toy, [[0.952574, 0.047426]], atol=1e-6), (
    f"small network: expected [[0.952574, 0.047426]], got {np.round(P_toy, 6)}. "
    "Its hidden layer outputs (1, 0) and its logits are (1, -2)")
P_mix = predict_proba(np.array([[1000.0, 0.0], [0.0, 0.0]]), toy)
assert np.all(np.isfinite(P_mix)), "a logit of 1000 gives nan or inf: subtract each row's own largest logit before exp (one max for the whole batch leaves 0/0 in rows far below it)"
assert np.allclose(P_mix[1], [0.731059, 0.268941], atol=1e-6), f"the second row should be the softmax of (0, -1), got {np.round(P_mix[1], 6)}"

recorded = np.array([
    [0.201049, 0.750163, 0.048788],
    [0.769190, 0.174410, 0.056400],
    [0.337658, 0.332001, 0.330341],
    [0.167320, 0.574679, 0.258002],
    [0.430389, 0.351689, 0.217922],
])
P = predict_proba(X_field, layers)
assert P.shape == (5, 3), f"expected shape (5, 3), got {P.shape}"
assert np.allclose(P.sum(axis=1), 1), "the probabilities of a reading should add up to 1"
assert np.allclose(P, recorded, atol=2e-6), (
    "the probabilities differ from the PyTorch run: check which layers get a ReLU and that the last layer's logits go into the softmax unchanged\n"
    f"{np.round(P, 6)}")
assert np.allclose(predict_proba(X_field[3:4], layers), P[3:4]), "one reading on its own should get the same probabilities as inside the batch"
P_big = predict_proba(400 * X_field, layers)
assert np.all(np.isfinite(P_big)), "readings 400 times larger give nan or inf: the logits overflow exp"
assert np.allclose(P_big.sum(axis=1), 1), "rows should still sum to 1 for large readings"
print("predicted classes", P.argmax(axis=1))
print("Part 2 passed")

## Solution

### Part 1: The layers in the lesson's layout

In [ ]:
def layers_from_state(state):
    """List of (W, b) in the order the network applies them, W of shape (n_in, n_out)."""
    def position(key):
        return int(key.split(".")[-2])

    layers = []
    for key in sorted((k for k in state if k.endswith(".weight")), key=position):
        W = np.asarray(state[key], dtype=float).T
        b = np.asarray(state[key[: -len("weight")] + "bias"], dtype=float)
        layers.append((W, b))
    return layers

**Why it works.** The keys sort as strings, so `net.10` lands between `net.0` and `net.2`.
Sorting by the integer position restores the order `nn.Sequential` applies the layers in.
PyTorch stores each weight as `(out_features, in_features)` and computes `x @ W.T + b`, so
the transpose gives the `(n_in, n_out)` matrix that the lesson's `A @ W + b` expects. Here
the four hidden weights are square, so a missing transpose would raise no error at all,
which is why the check compares values and not only shapes.

### Part 2: The probabilities for a batch

In [ ]:
def predict_proba(X, layers):
    """Class probabilities, one row per row of X."""
    A = np.asarray(X, dtype=float)
    for l, (W, b) in enumerate(layers, start=1):
        Z = A @ W + b
        if l < len(layers):
            A = np.maximum(0.0, Z)
        else:
            peak = np.max(Z, axis=1)[:, None]
            weights = np.exp(Z - peak)
            A = weights / weights.sum(axis=1)[:, None]
    return A

**Why it works.** The exported model is `Linear, ReLU, ..., Linear` with no ReLU after the
last `Linear`, so the logits can be negative and go into the softmax as they are. A ReLU
there would turn every negative logit into 0 and change the probabilities without raising
anything, as the check on the recorded values shows. Subtracting each row's largest logit
leaves the probabilities unchanged and keeps every exponent at or below 0, so the readings
scaled by 400, whose logits run into the hundreds, still give finite probabilities.